# 第 8 周练习：多智能体任务编排器（Multi-Agent Task Orchestrator）

## 练习目标（理念）

搭建一个 **多智能体（multi-agent）系统**：多个「专长不同」的 Agent 协作，完成复杂任务。

**用例：** 自治任务编排器——把复杂请求拆成子任务，分配给专长 Agent，再汇总（synthesize）成最终回答。

## 和本课 Week 8 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 第 1 天 Modal / SpecialistAgent | 多个 Specialist：Research / Code / Review / Writer |
| 第 2 天 RAG / Frontier / Ensemble | 分工协作的编排思路（本练习用 API Agent） |
| 第 4 天 AutonomousPlanningAgent + tool calling | `PlannerAgent` JSON 计划；后半段 `TOOLS` 函数调用 |
| 第 5 天 Deal Agent Framework | `TaskOrchestrator` 串起 plan → execute → synthesize |

## 架构（数据流）

```
+-------------------+
|  User Request     |
+-------------------+
         |
         v
+-------------------+
|  Planner Agent    |  <-- 拆任务、指派专长 Agent
+-------------------+
         |
    +----+----+----+
    |    |    |    |
    v    v    v    v
+------+ +------+ +------+ +------+
|Research| |Code | |Review| |Summary|
|Agent  | |Agent| |Agent | |Agent |
+------+ +------+ +------+ +------+
    |    |    |    |
    +----+----+----+
         |
         v
+-------------------+
|  Synthesizer      |  <-- 合并各 Agent 结果
+-------------------+
         |
         v
+-------------------+
|  Final Response   |
+-------------------+
```

## 怎么跑

1. 准备 `.env`：优先 `OPENROUTER_API_KEY`；否则回退到本机 `OPENAI_API_KEY`（OpenAI SDK 默认）
2. 从上到下运行；先跑测试单元格看日志，再 `demo.launch()` 打开 Gradio
3. 发给模型的 system/user prompt、tool description **保持英文**（改译会改变行为）

---


In [ ]:
# ========== 导入：环境、类型、OpenAI、Gradio、日志 ==========

# 标准库：环境变量、JSON、日志
import os
import json
import logging
# dataclass：用装饰器快速定义结果/计划等数据结构
from dataclasses import dataclass, field
# 类型标注：List / Dict / Optional / Callable，方便读接口
from typing import List, Dict, Optional, Callable
# load_dotenv：把 .env 读进环境变量，避免把密钥写进代码
from dotenv import load_dotenv
# OpenAI 客户端：后面走 OpenRouter 或官方 API
from openai import OpenAI
# Gradio：搭交互 UI
import gradio as gr

# override=True：.env 里的值覆盖进程里已有同名环境变量
load_dotenv(override=True)

# 配置 logging：INFO 级，后面各 Agent 执行时打进度
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


In [ ]:
# ========== 配置：OpenRouter / 模型名 / 初始化 client ==========

# OpenRouter 的 OpenAI 兼容 base URL
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

# 不同角色可换不同模型；这里先都用同一便宜模型，便于演示
PLANNER_MODEL = "openai/gpt-4.1-mini"      # 擅长规划（planning）
SPECIALIST_MODEL = "openai/gpt-4.1-mini"   # 专长任务
SYNTHESIZER_MODEL = "openai/gpt-4.1-mini"  # 擅长汇总（summarization）

# 优先读 OpenRouter Key；没有则回退官方 OpenAI，并改成非 OpenRouter 模型名
api_key = os.getenv("OPENROUTER_API_KEY")
if api_key:
    # 走 OpenRouter：base_url + 其模型命名空间（openai/...）
    client = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=api_key)
    print("Using OpenRouter API")
else:
    # 直接用 OpenAI 官方默认 endpoint；模型名去掉路由前缀
    client = OpenAI()
    PLANNER_MODEL = "gpt-4.1-mini"
    SPECIALIST_MODEL = "gpt-4.1-mini"
    SYNTHESIZER_MODEL = "gpt-4.1-mini"
    print("Using OpenAI API directly")


## Agent 基类（Base Class）

所有专长 Agent 都继承这个基类：统一 `__init__` 字段与 `execute()` 调用 Chat Completions 的路径。


In [ ]:
# ========== AgentResult + BaseAgent：统一执行与错误包装 ==========

@dataclass
class AgentResult:
    """单个 Agent 一次执行的结果（成功/失败都用同一结构）。"""
    agent_name: str
    task: str
    result: str
    success: bool = True
    # metadata 默认空 dict；field(default_factory=...) 避免可变默认值陷阱
    metadata: Dict = field(default_factory=dict)


class BaseAgent:
    """所有 Agent 的基类：持有 name / model / system_prompt，统一 execute。"""
    
    def __init__(self, name: str, model: str, system_prompt: str):
        # 展示名、所用模型、系统提示词（角色设定）
        self.name = name
        self.model = model
        self.system_prompt = system_prompt
    
    def execute(self, task: str) -> AgentResult:
        """执行该 Agent 的任务：调一次 chat.completions.create。"""
        try:
            # 日志只截前 50 字，避免刷屏
            logger.info(f"[{self.name}] Executing: {task[:50]}...")
            response = client.chat.completions.create(
                model=self.model,
                messages=[
                    # system：角色与行为约束（英文 prompt 勿改译）
                    {"role": "system", "content": self.system_prompt},
                    # user：具体子任务
                    {"role": "user", "content": task}
                ],
                temperature=0.7,
                max_tokens=1000
            )
            if response.choices:
                # 取出助手文本；None 时用空串
                result = response.choices[0].message.content or ""
                logger.info(f"[{self.name}] Completed successfully")
                return AgentResult(
                    agent_name=self.name,
                    task=task,
                    result=result,
                    success=True
                )
            # 有响应对象但没有 choices：当作失败
            return AgentResult(
                agent_name=self.name,
                task=task,
                result="No response from model",
                success=False
            )
        except Exception as e:
            # 网络/鉴权/解析等异常：记日志并返回 success=False
            logger.error(f"[{self.name}] Error: {e}")
            return AgentResult(
                agent_name=self.name,
                task=task,
                result=f"Error: {str(e)}",
                success=False
            )


## 专长 Agent（Specialized Agents）

每个子类只负责一件事：在 `__init__` 里写入不同的 `system_prompt`（英文角色设定），复用基类 `execute()`。


In [ ]:
# ========== 四个专长 Agent：Research / Code / Review / Writer ==========

class ResearchAgent(BaseAgent):
    """擅长调研与信息收集的 Agent。"""
    
    def __init__(self):
        # 调用基类：固定名字、模型、英文 system_prompt（影响回答风格，勿改译）
        super().__init__(
            name="Research Agent",
            model=SPECIALIST_MODEL,
            system_prompt="""You are a research specialist. Your role is to:
- Gather relevant information about topics
- Identify key facts, statistics, and insights
- Provide well-organized research summaries
- Cite sources when possible (even if simulated)

Be thorough but concise. Focus on accuracy and relevance."""
        )


class CodeAgent(BaseAgent):
    """擅长写代码与分析代码的 Agent。"""
    
    def __init__(self):
        super().__init__(
            name="Code Agent",
            model=SPECIALIST_MODEL,
            system_prompt="""You are a coding specialist. Your role is to:
- Write clean, efficient code
- Explain code functionality
- Debug and fix issues
- Suggest best practices

Always include code comments and explain your approach."""
        )


class ReviewAgent(BaseAgent):
    """擅长评审与挑刺的 Agent。"""
    
    def __init__(self):
        super().__init__(
            name="Review Agent",
            model=SPECIALIST_MODEL,
            system_prompt="""You are a critical review specialist. Your role is to:
- Evaluate quality and accuracy of content
- Identify strengths and weaknesses
- Suggest improvements
- Provide constructive feedback

Be fair, thorough, and constructive in your reviews."""
        )


class WriterAgent(BaseAgent):
    """擅长写作与润色的 Agent。"""
    
    def __init__(self):
        super().__init__(
            name="Writer Agent",
            model=SPECIALIST_MODEL,
            system_prompt="""You are a writing specialist. Your role is to:
- Create clear, engaging content
- Adapt tone and style to the audience
- Structure information logically
- Edit and polish text

Focus on clarity, engagement, and proper structure."""
        )


## Planner Agent（含规划 / 后续可接 Tool Calling）

Planner 负责把复杂请求拆成子任务，并指定交给哪个专长 Agent。


In [ ]:
# ========== PlannerAgent：让模型输出 JSON 计划，失败则 Fallback ==========

@dataclass
class TaskPlan:
    """一次复杂任务的执行计划。"""
    original_request: str
    subtasks: List[Dict]
    strategy: str


class PlannerAgent:
    """负责规划并编排其他 Agent 的规划者。"""
    
    # 可供指派的 Agent 名 → 用途说明（会嵌进 prompt）
    AVAILABLE_AGENTS = {
        "research": "For gathering information, facts, and context",
        "code": "For writing, analyzing, or debugging code",
        "review": "For critiquing, evaluating, and suggesting improvements",
        "writer": "For creating, editing, or polishing written content"
    }
    
    def __init__(self):
        self.name = "Planner Agent"
        self.model = PLANNER_MODEL
    
    def create_plan(self, request: str) -> TaskPlan:
        """针对给定请求生成 TaskPlan（JSON → 数据类）。"""
        # 把可用 Agent 列表格式化进 prompt
        agents_desc = "\n".join([f"- {k}: {v}" for k, v in self.AVAILABLE_AGENTS.items()])
        
        # 规划 prompt 保留英文：要求模型只返回 JSON
        prompt = f"""Analyze this request and create a plan to complete it using available agents.

REQUEST: {request}

AVAILABLE AGENTS:
{agents_desc}

Create a JSON plan with this structure:
{{
    "strategy": "Brief description of overall approach",
    "subtasks": [
        {{
            "agent": "agent_name",
            "task": "Specific task description",
            "order": 1
        }}
    ]
}}

Rules:
- Use 2-4 subtasks maximum
- Each subtask should be specific and actionable
- Order subtasks logically (some may run in parallel)
- Only use agents from the available list

Respond with ONLY the JSON, no other text."""
        
        try:
            response = client.chat.completions.create(
                model=self.model,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.3,
                max_tokens=500
            )
            if response.choices:
                raw = response.choices[0].message.content or "{}"
                # 去掉可能包着的 Markdown 代码围栏 ```json ... ```
                raw = raw.strip()
                if raw.startswith("```"):
                    raw = raw.split("```")[1]
                    if raw.startswith("json"):
                        raw = raw[4:]
                raw = raw.strip()
                
                # 解析 JSON 并填入 TaskPlan
                plan_data = json.loads(raw)
                return TaskPlan(
                    original_request=request,
                    subtasks=plan_data.get("subtasks", []),
                    strategy=plan_data.get("strategy", "Execute tasks sequentially")
                )
        except Exception as e:
            logger.error(f"Planning error: {e}")
        
        # Fallback：规划失败时退化成「只派 research 做整题」
        return TaskPlan(
            original_request=request,
            subtasks=[{"agent": "research", "task": request, "order": 1}],
            strategy="Fallback: Single agent execution"
        )


## Synthesizer Agent（汇总者）

把多个 Agent 的结果合成一份连贯的最终回答。


In [ ]:
# ========== SynthesizerAgent：多路结果 → 统一最终答复 ==========

class SynthesizerAgent:
    """把多个 Agent 的结果综合成最终回复。"""
    
    def __init__(self):
        self.name = "Synthesizer Agent"
        self.model = SYNTHESIZER_MODEL
    
    def synthesize(self, original_request: str, results: List[AgentResult]) -> str:
        """综合多个 AgentResult，生成面向用户的最终文本。"""
        
        # 把每个 Agent 的任务与结果拼成大段上下文
        results_text = "\n\n".join([
            f"=== {r.agent_name} ===\nTask: {r.task}\nResult:\n{r.result}"
            for r in results
        ])
        
        # 汇总 prompt 保留英文（要求不要点名 Agent，呈现统一叙事）
        prompt = f"""You are synthesizing results from multiple specialized agents to answer the user's request.

ORIGINAL REQUEST:
{original_request}

AGENT RESULTS:
{results_text}

Create a comprehensive, well-organized response that:
1. Directly addresses the original request
2. Integrates insights from all agents
3. Maintains a coherent narrative
4. Highlights key findings and recommendations

Do not mention the agents by name. Present the information as a unified response."""
        
        try:
            response = client.chat.completions.create(
                model=self.model,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.5,
                max_tokens=1500
            )
            if response.choices:
                return response.choices[0].message.content or "Synthesis failed"
        except Exception as e:
            logger.error(f"Synthesis error: {e}")
        
        # API 失败时的兜底：直接把各结果拼接返回
        return "\n\n".join([r.result for r in results])


## Task Orchestrator（编排器）

协调整条多智能体工作流：计划 → 执行子任务 → 汇总。


In [ ]:
# ========== TaskOrchestrator：plan → execute specialists → synthesize ==========

class TaskOrchestrator:
    """编排多智能体任务执行的总控。"""
    
    def __init__(self):
        # 规划者 + 汇总者 + 四个专长实例
        self.planner = PlannerAgent()
        self.synthesizer = SynthesizerAgent()
        self.agents = {
            "research": ResearchAgent(),
            "code": CodeAgent(),
            "review": ReviewAgent(),
            "writer": WriterAgent()
        }
        # 执行日志：给 UI / print 回放用
        self.execution_log: List[str] = []
    
    def log(self, message: str):
        """写入执行日志，同时打到 logger。"""
        self.execution_log.append(message)
        logger.info(message)
    
    def execute(self, request: str) -> Dict:
        """用多智能体流水线执行一次复杂任务。"""
        # 每次执行清空日志
        self.execution_log = []
        
        # 步骤 1：让 Planner 生成计划
        self.log(f"Received request: {request[:100]}...")
        self.log("Creating execution plan...")
        plan = self.planner.create_plan(request)
        self.log(f"Strategy: {plan.strategy}")
        self.log(f"Subtasks: {len(plan.subtasks)}")
        
        # 步骤 2：按 order 排序后依次执行各子任务
        results: List[AgentResult] = []
        for subtask in sorted(plan.subtasks, key=lambda x: x.get("order", 0)):
            agent_name = subtask.get("agent", "research")
            task = subtask.get("task", request)
            
            if agent_name in self.agents:
                self.log(f"Executing {agent_name} agent: {task[:50]}...")
                agent = self.agents[agent_name]
                result = agent.execute(task)
                results.append(result)
                self.log(f"{agent_name} agent completed: {'Success' if result.success else 'Failed'}")
            else:
                # 计划里出现未知 agent 名：跳过并记日志
                self.log(f"Unknown agent: {agent_name}, skipping...")
        
        # 步骤 3：汇总所有结果
        self.log("Synthesizing results...")
        final_response = self.synthesizer.synthesize(request, results)
        self.log("Synthesis complete!")
        
        # 返回结构化字典：计划、截断后的 agent 结果、最终答复、日志
        return {
            "request": request,
            "plan": {
                "strategy": plan.strategy,
                "subtasks": plan.subtasks
            },
            "agent_results": [
                {
                    "agent": r.agent_name,
                    "task": r.task,
                    # 结果太长时截断，避免 UI 爆炸
                    "result": r.result[:500] + "..." if len(r.result) > 500 else r.result,
                    "success": r.success
                }
                for r in results
            ],
            "final_response": final_response,
            "execution_log": self.execution_log
        }


## 测试编排器

用一条「研究 + 写代码 + 评审」的复合请求做冒烟测试。


In [ ]:
# ========== 冒烟测试：创建编排器并跑一条复合请求 ==========

# 创建编排器实例
orchestrator = TaskOrchestrator()

# 测试请求保持英文：会影响 Planner 拆任务与各 Agent 回答语言
test_request = """I need help creating a Python function that calculates the Fibonacci sequence.
Please research best practices, write the code, and review it for improvements."""

# 跑完整流水线
result = orchestrator.execute(test_request)

# 打印执行日志
print("=== EXECUTION LOG ===")
for log in result["execution_log"]:
    print(f"  {log}")

# 打印计划：策略 + 子任务列表
print("\n=== PLAN ===")
print(f"Strategy: {result['plan']['strategy']}")
for subtask in result['plan']['subtasks']:
    print(f"  [{subtask.get('order', 0)}] {subtask.get('agent')}: {subtask.get('task')[:60]}...")

# 打印最终汇总回答
print("\n=== FINAL RESPONSE ===")
print(result["final_response"])


## Gradio UI：交互式任务编排器

把编排器接到网页：输入请求，分栏查看最终答复 / 计划 / 各 Agent 结果 / 日志。


In [ ]:
# ========== Gradio 回调：process_request + 示例请求列表 ==========

# 全局编排器实例（UI 多次点击复用同一对象）
orchestrator = TaskOrchestrator()

def process_request(request: str) -> tuple:
    """处理一次请求，返回四个展示字符串：最终答复、计划、Agent 结果、日志。"""
    # 空输入直接提示，不调 API
    if not request.strip():
        return "Please enter a request.", "", "", ""
    
    result = orchestrator.execute(request)
    
    # 把计划格式化成 Markdown 表格
    plan_md = f"""### Strategy
{result['plan']['strategy']}

### Subtasks
| Order | Agent | Task |
|-------|-------|------|
"""
    for subtask in result['plan']['subtasks']:
        plan_md += f"| {subtask.get('order', 0)} | {subtask.get('agent')} | {subtask.get('task')[:50]}... |\n"
    
    # 把每个 Agent 结果拼成 Markdown 小节
    agents_md = ""
    for ar in result['agent_results']:
        status = "Success" if ar['success'] else "Failed"
        agents_md += f"""### {ar['agent']} ({status})
**Task:** {ar['task']}

{ar['result']}

---

"""
    
    # 日志加序号，便于对照
    log_text = "\n".join([f"[{i+1}] {log}" for i, log in enumerate(result['execution_log'])])
    
    return result['final_response'], plan_md, agents_md, log_text


# 示例请求（英文）：一键填入输入框，方便演示
EXAMPLE_REQUESTS = [
    "Write a Python function to check if a number is prime, with tests and documentation.",
    "Explain the concept of recursion in programming with examples and best practices.",
    "Create a simple REST API design for a todo list application.",
    "Research machine learning basics and write a beginner-friendly summary.",
    "Write a Python script to parse JSON files and review it for error handling."
]

def load_example(idx: int) -> str:
    """按索引加载示例请求；越界返回空串。"""
    if 0 <= idx < len(EXAMPLE_REQUESTS):
        return EXAMPLE_REQUESTS[idx]
    return ""


In [ ]:
# ========== 搭建并启动 Gradio Blocks 界面 ==========

# Soft 主题 + 标题；界面文案保留英文（Gradio 展示字符串）
with gr.Blocks(title="Multi-Agent Task Orchestrator", theme=gr.themes.Soft()) as demo:
    gr.Markdown("""# Multi-Agent Task Orchestrator
    
Enter a complex task and watch specialized agents collaborate to complete it.

**Available Agents:**
- **Research Agent**: Gathers information and context
- **Code Agent**: Writes and analyzes code
- **Review Agent**: Evaluates and suggests improvements
- **Writer Agent**: Creates and polishes content
""")
    
    # 左侧主输入区
    with gr.Row():
        with gr.Column(scale=2):
            request_input = gr.Textbox(
                label="Your Request",
                placeholder="Describe what you need help with...",
                lines=4
            )
            with gr.Row():
                submit_btn = gr.Button("Execute", variant="primary")
                clear_btn = gr.Button("Clear")
            
            # 示例按钮：闭包 idx=i 固定每颗按钮对应的示例
            gr.Markdown("### Example Requests:")
            with gr.Row():
                for i, ex in enumerate(EXAMPLE_REQUESTS[:3]):
                    btn = gr.Button(f"Example {i+1}", size="sm")
                    btn.click(fn=lambda idx=i: EXAMPLE_REQUESTS[idx], outputs=request_input)
    
    # 四个 Tab：最终答复 / 计划 / Agent 结果 / 日志
    with gr.Tabs():
        with gr.Tab("Final Response"):
            response_output = gr.Markdown(label="Response")
        
        with gr.Tab("Execution Plan"):
            plan_output = gr.Markdown(label="Plan")
        
        with gr.Tab("Agent Results"):
            agents_output = gr.Markdown(label="Agent Results")
        
        with gr.Tab("Execution Log"):
            log_output = gr.Textbox(label="Log", lines=15, interactive=False)
    
    # Execute：跑编排器并填充四个输出
    submit_btn.click(
        fn=process_request,
        inputs=request_input,
        outputs=[response_output, plan_output, agents_output, log_output]
    )
    # Clear：清空输入与全部输出
    clear_btn.click(
        fn=lambda: ("", "", "", "", ""),
        outputs=[request_input, response_output, plan_output, agents_output, log_output]
    )

# 启动 Gradio（本地默认端口）
demo.launch()


## 进阶：Tool-Calling Planner

更「工具化」的规划方式：用 OpenAI **function calling**，让模型通过工具名指派任务，而不是只吐 JSON 文本。


In [ ]:
# ========== TOOLS 定义 + handle_tool_calls：函数调用式任务分配 ==========

# 给 Planner 用的 tools schema（OpenAI tools 格式；description 影响模型选工具）
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "assign_research_task",
            "description": "Assign a research task to gather information",
            "parameters": {
                "type": "object",
                "properties": {
                    "topic": {"type": "string", "description": "The topic to research"}
                },
                "required": ["topic"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "assign_code_task",
            "description": "Assign a coding task to write or analyze code",
            "parameters": {
                "type": "object",
                "properties": {
                    "task": {"type": "string", "description": "The coding task description"}
                },
                "required": ["task"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "assign_review_task",
            "description": "Assign a review task to evaluate content",
            "parameters": {
                "type": "object",
                "properties": {
                    "content": {"type": "string", "description": "What to review"}
                },
                "required": ["content"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "assign_writing_task",
            "description": "Assign a writing task to create content",
            "parameters": {
                "type": "object",
                "properties": {
                    "task": {"type": "string", "description": "The writing task description"}
                },
                "required": ["task"]
            }
        }
    }
]

def handle_tool_calls(tool_calls) -> List[Dict]:
    """把模型返回的 tool_calls 转成编排器用的 subtasks 列表。"""
    tasks = []
    # 工具函数名 → 内部 agent 键
    agent_map = {
        "assign_research_task": "research",
        "assign_code_task": "code",
        "assign_review_task": "review",
        "assign_writing_task": "writer"
    }
    
    for i, call in enumerate(tool_calls):
        func_name = call.function.name
        # arguments 是 JSON 字符串，需 loads
        args = json.loads(call.function.arguments)
        # 不同工具参数字段名不同：topic / task / content
        task_desc = args.get("topic") or args.get("task") or args.get("content", "")
        
        if func_name in agent_map:
            tasks.append({
                "agent": agent_map[func_name],
                "task": task_desc,
                "order": i + 1
            })
    
    return tasks

# 打印已定义工具，确认 schema 加载成功
print("Tool-calling planner defined. Tools available:")
for tool in TOOLS:
    print(f"  - {tool['function']['name']}: {tool['function']['description']}")


## 摘要

本练习演示了：

1. **Agent 架构**：基类 + 多个专长实现
2. **规划（Planning）**：把复杂任务拆成子任务
3. **Tool Calling**：用 OpenAI function calling 做任务指派
4. **编排（Orchestration）**：协调多个 Agent 的执行顺序
5. **汇总（Synthesis）**：把多路结果合成连贯回答
6. **Gradio UI**：交互式操作整条流水线

### Week 8 关键概念对照

| 概念 | 本练习实现 |
|------|------------|
| Specialist Agents | `ResearchAgent` / `CodeAgent` / `ReviewAgent` / `WriterAgent` |
| Planning Agent | `PlannerAgent`（JSON 计划） |
| Tool Calling | `TOOLS` + `handle_tool_calls` |
| Orchestration | `TaskOrchestrator` 串起工作流 |
| Synthesis | `SynthesizerAgent` 合并结果 |

### 可扩展方向

1. **Modal 部署**：把 Agent 做成 serverless 函数
2. **RAG 集成**：加向量库做上下文检索
3. **并行执行**：无依赖子任务并发跑
4. **Memory**：跨会话保存对话历史
5. **Streaming**：流式展示各 Agent 输出
